# NIFTY 50 Portfolio Allocation System

This project builds a simple portfolio allocation system using Python.

The program:
- Collects live stock data from Yahoo Finance
- Extracts important company information
- Calculates equal-weight investment allocation
- Determines the number of shares to purchase
- Exports the final portfolio into a formatted Excel file

Technologies Used:
- Python
- Pandas
- NumPy
- yFinance API
- XlsxWriter

In [ ]:
import numpy as np
import pandas as pd
import requests
import xlsxwriter 
import yfinance as yf
import math

## Loading NIFTY 50 Stock Symbols

In this step, we import the list of NIFTY 50 companies from a CSV file.

The dataset contains stock symbols that will later be used to fetch real-time market data from Yahoo Finance.

In [ ]:
stocks = pd.read_csv("nifty_50_stocks.csv")
symbols= stocks["SYMBOL"]

## Creating the Portfolio DataFrame

A structured DataFrame is created to store:
- Stock Symbol
- Sector
- Current Stock Price
- Market Capitalization
- Number of Shares to Buy

This DataFrame will hold the final portfolio allocation results.

In [ ]:
my_col = [
    "Symbol" ,
    "Sector",
    "Stock Price" , 
    "Market Cap (Cr)", 

    "Number of shares to buy"
    ]
final_df = pd.DataFrame( columns=my_col)


## Fetching Live Market Data

We iterate through each stock symbol and use the Yahoo Finance API to collect:
- Current stock price
- Company sector
- Market capitalization

The collected information is then appended to the portfolio DataFrame.

In [ ]:

for stock in symbols:
    ticker = yf.Ticker(stock+ ".NS")
    info = ticker.info

    current_price = info["currentPrice"]

    sector = info["sector"]
    
    market_cap = round(info["marketCap"] / 10000000, 2)


    final_df.loc[len(final_df)] = [
        stock,
        sector,
        current_price,
        market_cap,

        "N/A"
    ]
print(final_df)

## User Portfolio Investment Input

The user is asked to enter the total investment amount for the portfolio.

Input validation is also performed to ensure the entered value is numeric and valid.

In [ ]:
portfolio_size = input("enter the size of your portfolio")
try : 
    val = float(portfolio_size)
    print("done got it")
except ValueError:
    print("that is not a number")

## Calculating Equal Weight Allocation

The total investment amount is divided equally among all stocks in the portfolio.

For each stock:
- Equal capital allocation is calculated
- The number of shares to buy is determined using the stock's current market price
- `math.floor()` is used to avoid fractional shares

In [ ]:
total_amount = val/len(final_df)
for i in range(0,len(final_df)):
    final_df.loc[i,"Number of shares to buy"] = math.floor(total_amount/final_df.loc[i,"Stock Price"])
print(final_df)


## Exporting Portfolio to Excel

The final portfolio is exported to an Excel file using XlsxWriter.

Additional formatting is applied to improve readability:
- Styled headers
- Currency formatting
- Adjusted column widths
- Professional spreadsheet layout

In [ ]:
writer = pd.ExcelWriter(
    "portfolio.xlsx",
    engine="xlsxwriter"
)

final_df.to_excel(
    writer,
    sheet_name="Portfolio",
    index=False
)

workbook = writer.book
worksheet = writer.sheets["Portfolio"]

header_format = workbook.add_format({
    "bold": True,
    "font_color": "white",
    "bg_color": "#4F81BD",
    "border": 1
})

money_format = workbook.add_format({
    "num_format": "₹#,##0.00"
})

percent_format = workbook.add_format({
    "num_format": "0.00%"
})

for col_num, value in enumerate(final_df.columns.values):

    worksheet.write(0, col_num, value, header_format)

worksheet.set_column("A:A", 15)
worksheet.set_column("B:B", 20)
worksheet.set_column("C:C", 15, money_format)
worksheet.set_column("D:D", 25, money_format)
worksheet.set_column("E:E", 20)

writer.close()
